# E-commerce Sales & Customer Analytics
## Data Investigation, Quality Assessment & Cleaning

**Objective:** prepare the Olist e-commerce dataset for downstream sales, customer, delivery, product, payment, review, RFM and churn-oriented analysis.

This notebook documents the investigation, conservative cleaning decisions, validation, and export of the nine source tables.

> Unusual values are investigated before changing them. Repeated keys can be legitimate because several tables are naturally one-to-many.

## 1. Load the Dataset

The source contains nine related tables: customers, sellers, orders, order items, products, payments, reviews, geolocation, and category translation.

In [ ]:
import os
import unicodedata
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

path = "/kaggle/input/datasets/olistbr/brazilian-ecommerce/"

customers = pd.read_csv(path + "olist_customers_dataset.csv")
sellers = pd.read_csv(path + "olist_sellers_dataset.csv")
reviews = pd.read_csv(path + "olist_order_reviews_dataset.csv")
order_items = pd.read_csv(path + "olist_order_items_dataset.csv")
products = pd.read_csv(path + "olist_products_dataset.csv")
geolocation = pd.read_csv(path + "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(path + "product_category_name_translation.csv")
orders = pd.read_csv(path + "olist_orders_dataset.csv")
payments = pd.read_csv(path + "olist_order_payments_dataset.csv")

tables = {
    "customers": customers,
    "sellers": sellers,
    "orders": orders,
    "order_items": order_items,
    "products": products,
    "payments": payments,
    "reviews": reviews,
    "geolocation": geolocation,
    "category_translation": category_translation
}

## 2. Initial Structure and Data Types

In [ ]:
overview = pd.DataFrame({
    "rows": {name: df.shape[0] for name, df in tables.items()},
    "columns": {name: df.shape[1] for name, df in tables.items()},
    "duplicate_rows": {name: int(df.duplicated().sum()) for name, df in tables.items()}
})
overview

In [ ]:
for name, df in tables.items():
    print(f"\n{name.upper()}")
    print("-" * 60)
    print(df.dtypes)

### Source dimensions

| Table | Rows | Columns |
|---|---:|---:|
| customers | 99,441 | 5 |
| sellers | 3,095 | 4 |
| orders | 99,441 | 8 |
| order_items | 112,650 | 7 |
| products | 32,951 | 9 |
| payments | 103,886 | 5 |
| reviews | 99,224 | 7 |
| geolocation | 1,000,163 | 5 |
| category_translation | 71 | 2 |

## 3. Missing Values

In [ ]:
missing_summary = pd.DataFrame({
    name: df.isna().sum()
    for name, df in tables.items()
}).T
missing_summary.loc[:, missing_summary.sum() > 0]

### Findings

- Customers, sellers, order items, payments, geolocation and category translation have no missing values.
- Review title/message fields contain many missing values and are valid absent-text cases.
- Products have 610 rows missing category/name/description/photo metadata, plus 2 rows with missing physical attributes.
- Orders have missing lifecycle timestamps. Their meaning depends on order status, so they are not automatically imputed.

## 4. Order Status and Lifecycle Dates

In [ ]:
orders["order_status"].value_counts()

In [ ]:
date_cols = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

orders.groupby("order_status")[date_cols].apply(lambda x: x.isna().sum())

In [ ]:
order_date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

date_ranges = []
for col in order_date_cols:
    s = pd.to_datetime(orders[col], errors="coerce")
    date_ranges.append([col, s.min(), s.max(), int(s.isna().sum())])

pd.DataFrame(date_ranges, columns=["column", "min_date", "max_date", "invalid_after_conversion"])

In [ ]:
orders_dates = orders.copy()
for col in order_date_cols:
    orders_dates[col] = pd.to_datetime(orders_dates[col], errors="coerce")

chronology = pd.Series({
    "approval_before_purchase": (orders_dates["order_approved_at"] < orders_dates["order_purchase_timestamp"]).sum(),
    "carrier_before_approval": (orders_dates["order_delivered_carrier_date"] < orders_dates["order_approved_at"]).sum(),
    "delivery_before_carrier": (orders_dates["order_delivered_customer_date"] < orders_dates["order_delivered_carrier_date"]).sum(),
    "delivery_before_purchase": (orders_dates["order_delivered_customer_date"] < orders_dates["order_purchase_timestamp"]).sum(),
    "estimated_before_purchase": (orders_dates["order_estimated_delivery_date"] < orders_dates["order_purchase_timestamp"]).sum()
})
chronology

### Date-quality findings

Date conversion produced no invalid dates. The main chronological exceptions are:

- 1,359 rows where the carrier timestamp precedes approval.
- 23 rows where customer delivery precedes the carrier timestamp.
- 0 rows where customer delivery precedes purchase.
- 0 rows where estimated delivery precedes purchase.

These source-level anomalies are documented rather than overwritten because the correct replacement timestamps cannot be established from the available columns.

In [ ]:
print("Delivered orders missing customer delivery date:",
      ((orders["order_status"] == "delivered") &
       orders["order_delivered_customer_date"].isna()).sum())

print("Delivered orders missing carrier date:",
      ((orders["order_status"] == "delivered") &
       orders["order_delivered_carrier_date"].isna()).sum())

## 5. Order Items

In [ ]:
print("Orders with more than one item:", (order_items["order_id"].value_counts() > 1).sum())
print("Maximum item number:", order_items["order_item_id"].max())
print("Duplicate order_id + order_item_id:", order_items.duplicated(["order_id", "order_item_id"]).sum())

print("\nPrice:")
print(order_items["price"].describe())

print("\nFreight:")
print(order_items["freight_value"].describe())

print("\nZero price:", (order_items["price"] == 0).sum())
print("Negative price:", (order_items["price"] < 0).sum())
print("Zero freight:", (order_items["freight_value"] == 0).sum())
print("Negative freight:", (order_items["freight_value"] < 0).sum())

In [ ]:
shipping_dates = pd.to_datetime(order_items["shipping_limit_date"], errors="coerce")
purchase_lookup = orders[["order_id", "order_purchase_timestamp"]].copy()
purchase_lookup["order_purchase_timestamp"] = pd.to_datetime(
    purchase_lookup["order_purchase_timestamp"], errors="coerce"
)

items_with_purchase = order_items.merge(purchase_lookup, on="order_id", how="left")

print("Invalid shipping dates:", shipping_dates.isna().sum())
print("Shipping limit before purchase:",
      (shipping_dates < items_with_purchase["order_purchase_timestamp"]).sum())
print("Shipping dates after 2019-01-01:", (shipping_dates > "2019-01-01").sum())

Multiple item rows per order are expected. The `order_id + order_item_id` combination is unique. Prices and freight contain no negative values.

Four shipping-limit timestamps extend beyond 2019, including 2020 records. They are retained because they exist in the source and are not enough by themselves to justify deletion.

## 6. Products and Category Coverage

In [ ]:
print("Missing product fields:")
print(products.isna().sum())

physical_cols = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

print("\nZero values:")
for col in physical_cols:
    print(col, (products[col] == 0).sum())

print("\nDuplicate product IDs:", products["product_id"].duplicated().sum())

In [ ]:
missing_category_products = products.loc[
    products["product_category_name"].isna(), "product_id"
]

sold_missing_category_items = order_items[
    order_items["product_id"].isin(missing_category_products)
]

print("Products without category:", len(missing_category_products))
print("Order-item rows involving them:", len(sold_missing_category_items))
print("Orders involving them:", sold_missing_category_items["order_id"].nunique())

In [ ]:
category_check = products[["product_category_name"]].drop_duplicates().merge(
    category_translation,
    on="product_category_name",
    how="left"
)

untranslated = category_check[
    category_check["product_category_name"].notna() &
    category_check["product_category_name_english"].isna()
]

print("Categories without English translation:", len(untranslated))
untranslated

In [ ]:
zero_weight_products = products[products["product_weight_g"] == 0]
zero_weight_items = order_items[
    order_items["product_id"].isin(zero_weight_products["product_id"])
]

print("Zero-weight products:", len(zero_weight_products))
print("Order-item rows:", len(zero_weight_items))
print("Orders:", zero_weight_items["order_id"].nunique())

### Product findings

610 products have missing category and related catalogue metadata. They still appear in 1,603 order-item rows across 1,451 orders, so they should not be discarded.

Four products have zero recorded weight and appear in eight order-item rows across six orders. Their records are retained rather than inventing replacement weights.

## 7. Payments

In [ ]:
print(payments["payment_type"].value_counts(dropna=False))

print("\nZero payment values:", (payments["payment_value"] == 0).sum())
print("Negative payment values:", (payments["payment_value"] < 0).sum())
print("Zero installments:", (payments["payment_installments"] == 0).sum())
print("Negative installments:", (payments["payment_installments"] < 0).sum())

print("\nPayment value:")
print(payments["payment_value"].describe())

In [ ]:
print("Payments without matching order:",
      (~payments["order_id"].isin(orders["order_id"])).sum())

print("Orders without payment:",
      (~orders["order_id"].isin(payments["order_id"])).sum())

print("\nnot_defined payment records:")
print(payments[payments["payment_type"] == "not_defined"])

In [ ]:
item_totals = order_items.groupby("order_id").agg(
    item_value=("price", "sum"),
    freight_value=("freight_value", "sum")
)
item_totals["item_plus_freight"] = item_totals["item_value"] + item_totals["freight_value"]

payment_totals = payments.groupby("order_id")["payment_value"].sum().rename("payment_total")

reconciliation = item_totals.join(payment_totals, how="inner")
reconciliation["difference"] = reconciliation["payment_total"] - reconciliation["item_plus_freight"]

print("Orders with absolute payment/order-value difference > 1:",
      (reconciliation["difference"].abs() > 1).sum())

reconciliation["difference"].describe()

Payment rows are naturally one-to-many. There are 9 zero-value payments, 2 zero-installment records, and 3 `not_defined` records. These are retained for traceability.

There are 249 orders where total payment differs from item price plus freight by more than 1. This is documented as a reconciliation issue rather than used as an automatic deletion rule.

## 8. Reviews

In [ ]:
print("Review score range:", reviews["review_score"].min(), "to", reviews["review_score"].max())
print("Invalid scores:",
      ((reviews["review_score"] < 1) | (reviews["review_score"] > 5)).sum())

print("Missing review titles:", reviews["review_comment_title"].isna().sum())
print("Missing review messages:", reviews["review_comment_message"].isna().sum())
print("Orders with multiple review rows:",
      (reviews["order_id"].value_counts() > 1).sum())
print("Complete duplicate rows:", reviews.duplicated().sum())

Review scores are valid from 1 to 5. Missing comments are preserved in the original columns. A separate cleaned text field will use `No comment` so later NLP/text analysis can distinguish missing text from actual comments.

## 9. Geolocation

In [ ]:
print("Shape:", geolocation.shape)
print("Complete duplicate rows:", geolocation.duplicated().sum())
print("Unique ZIP prefixes:", geolocation["geolocation_zip_code_prefix"].nunique())
print("ZIP prefixes with multiple rows:",
      (geolocation["geolocation_zip_code_prefix"].value_counts() > 1).sum())

invalid_lat = ((geolocation["geolocation_lat"] < -90) |
               (geolocation["geolocation_lat"] > 90)).sum()
invalid_lng = ((geolocation["geolocation_lng"] < -180) |
               (geolocation["geolocation_lng"] > 180)).sum()
zero_coordinates = ((geolocation["geolocation_lat"] == 0) |
                    (geolocation["geolocation_lng"] == 0)).sum()

print("Invalid latitude:", invalid_lat)
print("Invalid longitude:", invalid_lng)
print("Zero-coordinate rows:", zero_coordinates)

In [ ]:
zip_state_counts = geolocation.groupby(
    "geolocation_zip_code_prefix"
)["geolocation_state"].nunique()

zip_city_counts = geolocation.groupby(
    "geolocation_zip_code_prefix"
)["geolocation_city"].nunique()

print("ZIP prefixes mapped to multiple states:", (zip_state_counts > 1).sum())
print("ZIP prefixes mapped to multiple cities:", (zip_city_counts > 1).sum())

print("\nCity spelling variation example:")
print(
    geolocation["geolocation_city"]
    .value_counts()
    .loc[lambda s: s.index.str.contains("sao paulo|são paulo", case=False, na=False)]
)

The geolocation table contains 261,831 exact duplicate rows and many observations per ZIP prefix. This is different from a transactional duplicate: the table is a geographic mapping source.

There are no missing coordinates, invalid latitude/longitude values, or zero coordinates. City text contains formatting variations such as `sao paulo` and `são paulo`, so a normalized city field is useful for grouping.

## 10. Cross-table Referential Integrity

In [ ]:
integrity = pd.Series({
    "orders_missing_customer": (~orders["customer_id"].isin(customers["customer_id"])).sum(),
    "payments_missing_order": (~payments["order_id"].isin(orders["order_id"])).sum(),
    "reviews_missing_order": (~reviews["order_id"].isin(orders["order_id"])).sum(),
    "items_missing_order": (~order_items["order_id"].isin(orders["order_id"])).sum(),
    "items_missing_product": (~order_items["product_id"].isin(products["product_id"])).sum(),
    "items_missing_seller": (~order_items["seller_id"].isin(sellers["seller_id"])).sum()
}, name="missing_references")

integrity

In [ ]:
orders_without_items = orders[~orders["order_id"].isin(order_items["order_id"])]

print("Orders without items:", len(orders_without_items))
print("\nBy status:")
print(orders_without_items["order_status"].value_counts())

print("\nDelivered orders without items:",
      ((orders["order_status"] == "delivered") &
       ~orders["order_id"].isin(order_items["order_id"])).sum())

print("\nCustomer records:", customers["customer_id"].nunique())
print("Unique customer_unique_id:", customers["customer_unique_id"].nunique())
print("Customers with more than one order:",
      (orders["customer_id"].value_counts() > 1).sum())

### Relationship findings

The key transactional foreign-key checks show no missing references. There are 775 orders without item rows, concentrated in non-delivered statuses; there are 0 delivered orders without items.

`customer_id` and `customer_unique_id` are different concepts. The dataset contains 99,441 customer records but 96,096 unique customer identifiers, so later customer-level analysis should choose the correct identifier deliberately.

## 11. Cleaning Strategy

The cleaning is intentionally conservative:

1. Convert date columns to `datetime64[ns]`.
2. Convert numeric fields explicitly.
3. Strip leading/trailing whitespace from location and category text.
4. Normalize payment types to lowercase.
5. Preserve source missing values when they have analytical meaning.
6. Add separate analysis-friendly fields rather than overwriting source fields.
7. Do not delete one-to-many records, unusual timestamps, zero payments, zero weight, or payment differences without stronger evidence.

In [ ]:
customers_clean = customers.copy()
sellers_clean = sellers.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
products_clean = products.copy()
payments_clean = payments.copy()
reviews_clean = reviews.copy()
geolocation_clean = geolocation.copy()
category_translation_clean = category_translation.copy()

### 12. Customers and Sellers

In [ ]:
for df, city_col, state_col, zip_col in [
    (customers_clean, "customer_city", "customer_state", "customer_zip_code_prefix"),
    (sellers_clean, "seller_city", "seller_state", "seller_zip_code_prefix")
]:
    df[city_col] = df[city_col].str.strip()
    df[state_col] = df[state_col].str.strip()
    df[zip_col] = pd.to_numeric(df[zip_col], errors="coerce").astype("Int64")

### 13. Orders and Order Items

In [ ]:
for col in order_date_cols:
    orders_clean[col] = pd.to_datetime(orders_clean[col], errors="coerce")

orders_clean["order_status"] = orders_clean["order_status"].str.strip().str.lower()

order_items_clean["shipping_limit_date"] = pd.to_datetime(
    order_items_clean["shipping_limit_date"], errors="coerce"
)

order_items_clean["order_item_id"] = pd.to_numeric(
    order_items_clean["order_item_id"], errors="coerce"
).astype("Int64")

for col in ["price", "freight_value"]:
    order_items_clean[col] = pd.to_numeric(
        order_items_clean[col], errors="coerce"
    )

### 14. Products and Category Translation

In [ ]:
product_numeric_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in product_numeric_cols:
    products_clean[col] = pd.to_numeric(products_clean[col], errors="coerce")

products_clean["product_category_name"] = products_clean["product_category_name"].str.strip()

products_clean["product_category_name_clean"] = (
    products_clean["product_category_name"].fillna("Unknown")
)

category_translation_clean["product_category_name"] = (
    category_translation_clean["product_category_name"].str.strip()
)
category_translation_clean["product_category_name_english"] = (
    category_translation_clean["product_category_name_english"].str.strip()
)

### 15. Payments

In [ ]:
payments_clean["payment_sequential"] = pd.to_numeric(
    payments_clean["payment_sequential"], errors="coerce"
).astype("Int64")

payments_clean["payment_installments"] = pd.to_numeric(
    payments_clean["payment_installments"], errors="coerce"
).astype("Int64")

payments_clean["payment_value"] = pd.to_numeric(
    payments_clean["payment_value"], errors="coerce"
)

payments_clean["payment_type"] = (
    payments_clean["payment_type"].str.strip().str.lower()
)

### 16. Reviews

In [ ]:
reviews_clean["review_creation_date"] = pd.to_datetime(
    reviews_clean["review_creation_date"], errors="coerce"
)
reviews_clean["review_answer_timestamp"] = pd.to_datetime(
    reviews_clean["review_answer_timestamp"], errors="coerce"
)

reviews_clean["review_comment_title"] = reviews_clean["review_comment_title"].str.strip()
reviews_clean["review_comment_message"] = reviews_clean["review_comment_message"].str.strip()

reviews_clean["review_comment_title_clean"] = (
    reviews_clean["review_comment_title"].fillna("No comment")
)
reviews_clean["review_comment_message_clean"] = (
    reviews_clean["review_comment_message"].fillna("No comment")
)

### 17. Geolocation

In [ ]:
def normalize_city(value):
    if pd.isna(value):
        return value
    value = str(value).strip().lower()
    return "".join(
        c for c in unicodedata.normalize("NFKD", value)
        if not unicodedata.combining(c)
    )

geolocation_clean["geolocation_city"] = geolocation_clean["geolocation_city"].str.strip()
geolocation_clean["geolocation_state"] = geolocation_clean["geolocation_state"].str.strip()
geolocation_clean["geolocation_city_clean"] = (
    geolocation_clean["geolocation_city"].map(normalize_city)
)

## 18. Post-cleaning Validation

In [ ]:
clean_tables = {
    "customers": customers_clean,
    "sellers": sellers_clean,
    "orders": orders_clean,
    "order_items": order_items_clean,
    "products": products_clean,
    "payments": payments_clean,
    "reviews": reviews_clean,
    "geolocation": geolocation_clean,
    "category_translation": category_translation_clean
}

validation = pd.DataFrame({
    "rows": {name: df.shape[0] for name, df in clean_tables.items()},
    "columns": {name: df.shape[1] for name, df in clean_tables.items()},
    "duplicate_rows": {name: int(df.duplicated().sum()) for name, df in clean_tables.items()},
    "missing_cells": {name: int(df.isna().sum().sum()) for name, df in clean_tables.items()}
})

validation

In [ ]:
print("Remaining missing values:")
for name, df in clean_tables.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]
    print(f"\n{name}:")
    print("No missing values" if len(missing) == 0 else missing)

Remaining missing values are intentional source information rather than automatically deleted errors. In particular, order lifecycle dates depend on status, product metadata is incomplete for some products, and original review comment columns remain nullable.

The added cleaned fields provide explicit placeholders where that is useful for downstream analysis.

## 19. Export Cleaned Tables

In [ ]:
output_files = {
    "customers": "customers_clean.csv",
    "sellers": "sellers_clean.csv",
    "orders": "orders_clean.csv",
    "order_items": "order_items_clean.csv",
    "products": "products_clean.csv",
    "payments": "payments_clean.csv",
    "reviews": "reviews_clean.csv",
    "geolocation": "geolocation_clean.csv",
    "category_translation": "category_translation_clean.csv"
}

for name, filename in output_files.items():
    clean_tables[name].to_csv(filename, index=False)

for filename in output_files.values():
    print(filename, f"({os.path.getsize(filename):,} bytes)")

## 20. Final Data-Preparation Summary

The dataset is ready for the next analytical stage.

### Important findings documented in this notebook

- Main transactional tables contain no complete duplicate rows.
- Geolocation contains 261,831 exact duplicate rows and many observations per ZIP prefix.
- Review scores are valid from 1 to 5.
- Order-item prices and freight have no negative values.
- 610 products lack category and related catalogue metadata.
- Four products have zero recorded weight.
- Payment records include one-to-many structures, 9 zero-value payments, 2 zero-installment records, and 3 `not_defined` records.
- 1,359 carrier-before-approval and 23 delivery-before-carrier timestamp anomalies were identified.
- 775 orders have no item rows; none are delivered orders.
- Customer analysis should distinguish `customer_id` from `customer_unique_id`.

### Next stage

Use the cleaned tables for:
1. sales and revenue analysis
2. customer purchasing behaviour
3. delivery performance
4. product/category performance
5. seller performance
6. review analysis
7. RFM segmentation and churn-oriented feature engineering

> **Cleaning principle:** improve consistency without deleting source information that may be analytically meaningful.